# CEOPRO AI — In-Process RAG Chat Notebook

This bypasses `uvicorn`/FastAPI/HTTP entirely — it calls the RAG pipeline (`llm_client.answer_query()`) directly in this notebook's own Python process. There is no `localhost` socket involved anywhere, so this sidesteps the Windows loopback/connection issue completely.

**Run order:** run the *Setup* cell once, then run the *Chat* cell — it's a loop; type `exit` to stop it and get the cell back (re-run it any time to keep chatting).

**About the GROQ_API_KEY prompt in the Setup cell:** it uses `getpass`, which hides your typed input and does **not** save it into the notebook's cell source — so the key never ends up on disk in this file, even if you save the notebook afterward. Nothing else in this notebook touches or stores your key.

In [ ]:
import os
import sys

# Resolve the repo root regardless of where Jupyter's cwd ends up (launching
# from the repo root is normal, but some setups start elsewhere).
REPO_ROOT = os.getcwd()
if not os.path.isdir(os.path.join(REPO_ROOT, "src", "ai")):
    REPO_ROOT = r"C:\CEO PRO\CEOPRO-AI\.claude\worktrees\competitors-market-sentiment-a0ba02"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# --- Install dependencies for a true 1-click run. Installs from src/ai/requirements.txt
# (the project's own already-vetted, version-pinned dependency list) rather than a
# hand-picked/duplicated package list here, so this notebook can't silently drift out of
# sync with what the rest of the codebase actually declares and has been tested against.
# It's a superset of what this notebook strictly needs (also covers extraction/forecasting/
# pricing/the HTTP server this notebook otherwise avoids) - a bit heavier than the bare
# minimum, but correct and maintenance-free. Safe to re-run: pip no-ops on anything already
# satisfied, so this is fast on every run after the first. If imports below fail right after
# a first-time install, restart the kernel and re-run this cell once (standard Jupyter
# behavior - packages installed via !pip mid-session aren't always picked up without it).
%pip install -q -r "{REPO_ROOT}/src/ai/requirements.txt"
%pip install -q python-dotenv

import getpass

from dotenv import load_dotenv

# --- Local disposable test stack. Loaded from .env at the repo root
# (gitignored - never committed) - no credential values live in this
# notebook's own saved source. Fails loudly if .env is missing/incomplete
# rather than silently falling back to a hardcoded value. ---
load_dotenv(os.path.join(REPO_ROOT, ".env"))
_required = ["DATABASE_URL", "APP_DB_PASSWORD", "MINIO_ENDPOINT", "MINIO_ROOT_USER", "MINIO_ROOT_PASSWORD"]
_missing = [v for v in _required if not os.getenv(v)]
if _missing:
    raise RuntimeError(
        f"Missing from .env: {', '.join(_missing)}. Create a .env file at the repo root "
        "with these variables - see .env.example for the format."
    )

print("All dependencies installed.")

# --- Your real GROQ_API_KEY: typed input is hidden and is never written into
# this notebook's saved source. Leave blank and press Enter to skip (retrieval
# still works; the final generation step will raise LLMError until this is set). ---
os.environ["GROQ_API_KEY"] = getpass.getpass("Paste your GROQ_API_KEY (hidden, not saved to this file): ") or ""

# --- Tenant to query — defaults to the last verified end-to-end run's tenant
# (already has the real Impact Battery scrape and 109 forecasted products
# ingested into its RAG knowledge base). Change these if you want a different run. ---
TENANT_ID = "68bf13ff-1782-4814-a7fa-1d370ae1c223"
USER_ID = "8cd536f2-a9f9-464d-98b0-69a27852e978"

from src.ai import db
from src.ai.rag import llm_client as rag_llm_client

conn = db.app_role_connection(TENANT_ID, USER_ID)
print(f"Connected directly (no HTTP/socket layer) as tenant {TENANT_ID}.")
print("The first question in the Chat cell will load the embedding + reranker models (~20-40s, one-time); fast after that.")

## Chat

Run this cell, type a question, press Enter. Type `exit` or `quit` to stop the loop and get the cell prompt back — re-run the cell to start chatting again (no need to re-run Setup).

In [ ]:
TOP_K = 5

while True:
    query_text = input("You: ").strip()
    if not query_text:
        continue
    if query_text.lower() in ("exit", "quit"):
        print("Stopped. Re-run this cell to chat again.")
        break

    try:
        result = rag_llm_client.answer_query(conn, TENANT_ID, query_text, top_k=TOP_K)
        conn.commit()
    except rag_llm_client.LLMError as e:
        conn.rollback()
        print(f"LLM call failed: {e}\n")
        continue
    except Exception as e:
        conn.rollback()
        print(f"Query failed: {e}\n")
        continue

    print(f"\nAssistant: {result['answer']}\n")
    if result["sources"]:
        print("Sources:")
        for source in result["sources"]:
            print(f"  [{source['source_index']}] chunk={source['chunk_id']} score={source['score']}")
    print()

## When you're done

Clear the Setup cell's output (it may have echoed connection info, never the key itself) via *Cell → All Output → Clear*, then save. Optionally run `conn.close()` below, or just shut down the kernel.

In [ ]:
conn.close()
print("Connection closed.")